In [1]:
# !pip install "gymnasium[toy-text]"

In [1]:
import gymnasium as gym
from pyparsing import actions

env = gym.make('FrozenLake-v1', desc=None, map_name="4x4", is_slippery=True, render_mode="human")
init_state = env.reset()
print(init_state)
env.render()

(0, {'prob': 1})


In [2]:
print(env.observation_space, env.action_space)
action_to_symbol = {
    0: '\u2190',
    1: '\u2193',
    2: '\u2192',
    3: '\u2191'
}
for a in action_to_symbol.keys():
    print(a,':', action_to_symbol[a])

Discrete(16) Discrete(4)
0 : ←
1 : ↓
2 : →
3 : ↑


In [3]:
import jax.numpy as jnp
import jax

import numpy as np
class MDP():
    def __init__(self, env):
        self.states = np.arange(env.observation_space.n)
        self.actions = np.arange(env.action_space.n)
        self.P = env.unwrapped.P #pytree
        
mdp = MDP(env)

In [4]:
def init_v():
    V = jnp.zeros_like(mdp.states)
    def thunk(state):
        return V[state]
    return thunk

v_init = init_v()

In [6]:
from functools import partial
q_update = lambda  v_fn, state, action: np.sum([probs * (rewards + (1 -  dones) * v_fn(next_states)) for (probs, next_states, rewards, dones) in mdp.P[state][action]])
v_update =          lambda v_fn, state: np.max([q_update(v_fn, state, a) for a in mdp.actions])
def update(v_fn):
    V  = [v_update(v_fn, s) for s in mdp.states]
    return lambda state: V[state]

In [7]:
def value_iteration(v_fn, error=1e-5):
    v_fn_new = update(v_fn)
    if np.max(np.abs(np.array(list(map(v_fn_new, mdp.states))) - np.array(list(map(v_fn, mdp.states))))) < error:
        return v_fn_new
    else:
        return value_iteration(v_fn_new, error)
    
v_fn  = value_iteration(v_init)   

In [8]:
np.array(list(map(v_fn, mdp.states))).reshape((4,4))

array([[0.82329454, 0.82321581, 0.8231599 , 0.8231309 ],
       [0.82331164, 0.        , 0.52924383, 0.        ],
       [0.82334458, 0.82339098, 0.7645838 , 0.        ],
       [0.        , 0.88225465, 0.94112547, 0.        ]])

In [9]:
from jax import lax
# q_update = lambda  v, state, action: jnp.sum(jnp.array([probs * (rewards + (1 -  dones) * v[next_states]) for (probs, next_states, rewards, dones) in mdp.P[state][action]]))
q_update = lambda  v, state, action: jnp.sum(jnp.array([probs * (rewards + (1 -  dones) * v[next_states]) for (probs, next_states, rewards, dones) in mdp.P[state][action]]))
v_update =  lambda v, state: jnp.max(jnp.array([q_update(v, state, a) for a in mdp.actions]))
update  = lambda v: jnp.array([v_update(v, s) for s in mdp.states])
v_init = jnp.zeros_like(mdp.states, dtype=jnp.float32)
def value_iteration(v_init_, error=1e-5):
    updated_v_init = update(v_init_)
    def cond(vs):  
        v, updated_v = vs
        return jnp.max(jnp.abs(jnp.array(v) - jnp.array(updated_v) ))>= error
    def body(vs):
        v, updated_v = vs
        return (updated_v, update(updated_v))
    return lax.while_loop(cond, body, (v_init_, updated_v_init))[1]

v_solution  = value_iteration(v_init)   
v_solution.reshape((4,4))

Array([[0.82329607, 0.8232176 , 0.8231618 , 0.8231328 ],
       [0.82331306, 0.        , 0.5292447 , 0.        ],
       [0.8233458 , 0.8233919 , 0.76458454, 0.        ],
       [0.        , 0.88225543, 0.941126  , 0.        ]], dtype=float32)

In [10]:
env = gym.make("Taxi-v3", render_mode="human")
env.reset()
env.render()
env.observation_space

Discrete(500)